# Dual Conformal Calibration for LLM-VaR and LLM-ES

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/Conformal_Oracle/blob/main/CO_conformal_llm/CO_conformal_llm.ipynb)

This notebook demonstrates the **conformal prediction framework** for recalibrating
LLM-based VaR and ES forecasts from GPT-3.5, GPT-4, and GPT-4o.

**Paper:** *Distribution-Free Recalibration of Tail Quantile Forecasts under Temporal Dependence*

**Repository:** [QuantLet/Conformal_Oracle](https://github.com/QuantLet/Conformal_Oracle)


## 1. Setup and Data Loading


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuration
ALPHA_VAR = 0.01
ALPHA_ES  = 0.025
F_CAL     = 0.70
N_SAMPLES = 1024

ASSETS = ['CRIX', 'SP500', 'SPGTCLTR', 'stoxx', 'cact', 'gdaxi', 'cbu', 'ftse', 'djci']
WINDOWS = [30, 45, 60, 90, 120, 150]
MODELS = ['GPT-3.5', 'GPT-4', 'GPT-4o']

print(f'Models: {MODELS}')
print(f'Assets: {len(ASSETS)}, Windows: {len(WINDOWS)}')
print(f'Total combinations: {len(MODELS) * len(ASSETS) * len(WINDOWS)}')


Models: ['GPT-3.5', 'GPT-4', 'GPT-4o']
Assets: 9, Windows: 6
Total combinations: 162


## 2. Conformal Calibration Algorithm

**Algorithm (Dual Conformal Calibration):**

1. Split simulation outputs temporally: first 70% calibration, last 30% test
2. Compute VaR and ES from empirical quantiles of LLM samples
3. **VaR correction:** one-sided nonconformity score $s_t^V = \hat{q}_t^{lo} - r_t$
4. Compute $\hat{q}_V$ = $(1-\alpha)$-quantile of calibration scores
5. Corrected VaR: $\text{VaR}_t^{cp} = \text{VaR}_t^{raw} + \hat{q}_V$
6. **ES correction:** residual scores on violation days
7. Evaluate: Kupiec POF test, Basel Traffic Light, Acerbi $Z_2$


## 3. Key Results (w=30, mean across 9 assets)


In [2]:
# Pre-computed results from run_conformal_analysis_v2.py
results_w30 = pd.DataFrame({
    'Model': ['GPT-3.5', 'GPT-4', 'GPT-4o'],
    'Raw pi_hat': [0.004, 0.086, 0.057],
    'q_hat_V': [0.002, 0.024, 0.020],
    'q_hat_E': [-0.002, 0.021, 0.009],
    'Corr pi_hat': [0.006, 0.002, 0.002],
    'Green/9': ['8/9', '9/9', '9/9'],
    'Z2 Pass/9': ['9/9', '9/9', '9/9'],
})
print(results_w30.to_string(index=False))
print()
print('Key finding: GPT-4 raw violation rate 8.6% (Red Zone) -> 0.2% (Green Zone)')
print('Conformal correction q_V = +0.024 for GPT-4 vs +0.004 for GARCH-N')


  Model  Raw pi_hat  q_hat_V  q_hat_E  Corr pi_hat Green/9 Z2 Pass/9
 GPT-3.5       0.004    0.002   -0.002        0.006     8/9       9/9
   GPT-4       0.086    0.024    0.021        0.002     9/9       9/9
  GPT-4o       0.057    0.020    0.009        0.002     9/9       9/9

Key finding: GPT-4 raw violation rate 8.6% (Red Zone) -> 0.2% (Green Zone)
Conformal correction q_V = +0.024 for GPT-4 vs +0.004 for GARCH-N


## 4. Overall Results: 162 Combinations

| Metric | Value |
|---|---|
| Total combinations | 162 (3 models x 9 assets x 6 windows) |
| Basel Green Zone (corrected) | 144/162 (89%) |
| GPT-4/4o Green Zone | 108/108 (100%) |
| Z2 ES pass | 161/162 (99.4%) |
| Mean corrected pi_hat | 0.3% (target: 1%) |

The 18 non-Green combinations occur for GPT-3.5 at longer windows
where over-conservative distributions occasionally over-narrow.


## 5. Reproducing Full Results

To reproduce the full analysis, clone the repository and run:

```bash
git clone https://github.com/QuantLet/Conformal_Oracle.git
cd Conformal_Oracle
python run_conformal_analysis.py
```

This requires the LLM simulation CSV files (available on request from the authors)
and asset data Excel files.
